In [2]:
from PIL import Image

def apply_strong_jpeg_compression(image_path, output_path=None, quality=10):
    """
    Apply strong JPEG compression to an image.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the compressed image. If None, saves as '{original_name}_compressed.jpg'
    quality : int, optional
        JPEG quality factor (1-95). Lower values = more compression. Default is 10 (very low quality)
    
    Returns:
    --------
    str
        Path to the compressed image file
    """
    # Open the image
    img = Image.open(image_path)
    
    # Convert to RGB if necessary (JPEG doesn't support transparency)
    if img.mode in ('RGBA', 'LA', 'P'):
        rgb_img = Image.new('RGB', img.size, (255, 255, 255))
        if img.mode == 'P':
            img = img.convert('RGBA')
        rgb_img.paste(img, mask=img.split()[-1] if img.mode in ('RGBA', 'LA') else None)
        img = rgb_img
    elif img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_compressed.jpg"
    
    # Save with strong compression (low quality)
    img.save(output_path, 'JPEG', quality=quality, optimize=False)
    
    print(f"Compressed image saved to: {output_path}")
    print(f"Quality factor: {quality}")
    
    return output_path



In [6]:
# Example usage:


apply_strong_jpeg_compression('./figures/watermarked_img.png', './figures/watermarked_img_compressed.jpg', quality=8)
# apply_strong_jpeg_compression('photo.jpg')  # Uses default quality=10

Compressed image saved to: ./figures/watermarked_img_compressed.jpg
Quality factor: 8


'./figures/watermarked_img_compressed.jpg'

In [9]:
from PIL import Image

def apply_rescaling(image_path, output_path=None, scale_factor=0.5, interpolation='bilinear'):
    """
    Apply rescaling transformation (downscale + upscale) to test interpolation robustness.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the rescaled image. If None, saves as '{original_name}_rescaled.png'
    scale_factor : float, optional
        Factor to downscale by (0-1). Default is 0.5 (50% of original size)
    interpolation : str, optional
        Interpolation method: 'nearest', 'bilinear', 'bicubic', 'lanczos'
        Default is 'bilinear'
    
    Returns:
    --------
    str
        Path to the rescaled image file
    """
    # Mapping of interpolation methods
    interp_methods = {
        'nearest': Image.NEAREST,
        'bilinear': Image.BILINEAR,
        'bicubic': Image.BICUBIC,
        'lanczos': Image.LANCZOS
    }
    
    if interpolation.lower() not in interp_methods:
        raise ValueError(f"Invalid interpolation method. Choose from: {list(interp_methods.keys())}")
    
    resample_method = interp_methods[interpolation.lower()]
    
    # Open the image
    img = Image.open(image_path)
    original_size = img.size
    
    # Calculate downscaled size
    new_width = int(original_size[0] * scale_factor)
    new_height = int(original_size[1] * scale_factor)
    downscaled_size = (new_width, new_height)
    
    # Downscale
    img_downscaled = img.resize(downscaled_size, resample=resample_method)
    
    # Upscale back to original size
    img_rescaled = img_downscaled.resize(original_size, resample=resample_method)
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_rescaled.png"
    
    # Save without compression (PNG format preserves quality)
    img_rescaled.save(output_path, 'PNG')
    
    print(f"Rescaled image saved to: {output_path}")
    print(f"Original size: {original_size}")
    print(f"Downscaled to: {downscaled_size}")
    print(f"Upscaled back to: {original_size}")
    print(f"Interpolation method: {interpolation}")
    
    return output_path


# Example usage:
# apply_rescaling('input.png')  # Default: 50% downscale with bilinear
# apply_rescaling('input.png', scale_factor=0.25)  # 25% downscale
# apply_rescaling('input.png', interpolation='bicubic')  # Different interpolation

In [14]:
apply_rescaling('./figures/watermarked_img.png', './figures/watermarked_rescaling.png', scale_factor=0.25, interpolation='nearest')


Rescaled image saved to: ./figures/watermarked_rescaling.png
Original size: (1815, 1795)
Downscaled to: (453, 448)
Upscaled back to: (1815, 1795)
Interpolation method: nearest


'./figures/watermarked_rescaling.png'

In [15]:
from PIL import Image, ImageFilter

def apply_gaussian_blur(image_path, output_path=None, radius=5.0):
    """
    Apply Gaussian blur to simulate optical defocus or motion blur.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the blurred image. If None, saves as '{original_name}_blurred.png'
    radius : float, optional
        Blur radius (standard deviation). Higher values = more blur.
        Default is 5.0
        - Light blur: 1-3
        - Moderate blur: 3-7
        - Heavy blur: 7-15
        - Extreme blur: 15+
    
    Returns:
    --------
    str
        Path to the blurred image file
    """
    # Open the image
    img = Image.open(image_path)
    
    # Apply Gaussian blur
    img_blurred = img.filter(ImageFilter.GaussianBlur(radius=radius))
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_blurred.png"
    
    # Save without compression (PNG format)
    img_blurred.save(output_path, 'PNG')
    
    print(f"Blurred image saved to: {output_path}")
    print(f"Blur radius: {radius}")
    
    return output_path


In [17]:
# Example usage:
apply_gaussian_blur('./figures/watermarked_img.png',  radius=10.0)  # Default: radius=5.0
# apply_gaussian_blur('input.png', radius=10.0)  # Heavy blur
# apply_gaussian_blur('input.png', 'output.png', radius=2.0)  # Light blur

Blurred image saved to: ./figures/watermarked_img_blurred.png
Blur radius: 10.0


'./figures/watermarked_img_blurred.png'

In [ ]:
from PIL import Image
import random

def apply_random_resized_crop(image_path, output_path=None, crop_scale=(0.5, 1.0), 
                              aspect_ratio=(0.75, 1.33), interpolation='bilinear', seed=None):
    """
    Apply random resized cropping to simulate reframing or partial viewing of the image.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the cropped image. If None, saves as '{original_name}_cropped.png'
    crop_scale : tuple, optional
        Range of crop area relative to original image (min, max).
        Default is (0.5, 1.0) meaning crop 50-100% of the image
    aspect_ratio : tuple, optional
        Range of aspect ratio (width/height) for the crop (min, max).
        Default is (0.75, 1.33) for reasonable variations
    interpolation : str, optional
        Interpolation method: 'nearest', 'bilinear', 'bicubic', 'lanczos'
        Default is 'bilinear'
    seed : int, optional
        Random seed for reproducibility. If None, uses random values
    
    Returns:
    --------
    tuple
        (output_path, crop_info) where crop_info contains the crop parameters used
    """
    # Set random seed if provided
    if seed is not None:
        random.seed(seed)
    
    # Mapping of interpolation methods
    interp_methods = {
        'nearest': Image.NEAREST,
        'bilinear': Image.BILINEAR,
        'bicubic': Image.BICUBIC,
        'lanczos': Image.LANCZOS
    }
    
    if interpolation.lower() not in interp_methods:
        raise ValueError(f"Invalid interpolation method. Choose from: {list(interp_methods.keys())}")
    
    resample_method = interp_methods[interpolation.lower()]
    
    # Open the image
    img = Image.open(image_path)
    original_size = img.size
    width, height = original_size
    area = width * height
    
    # Randomly select crop area scale
    target_area = area * random.uniform(crop_scale[0], crop_scale[1])
    
    # Randomly select aspect ratio
    log_ratio = (random.uniform(aspect_ratio[0], aspect_ratio[1]))
    
    # Calculate crop dimensions
    crop_width = int(round((target_area * log_ratio) ** 0.5))
    crop_height = int(round((target_area / log_ratio) ** 0.5))
    
    # Ensure crop dimensions don't exceed original image
    if crop_width > width:
        crop_width = width
    if crop_height > height:
        crop_height = height
    
    # Randomly select crop position
    left = random.randint(0, width - crop_width)
    top = random.randint(0, height - crop_height)
    right = left + crop_width
    bottom = top + crop_height
    
    # Perform the crop
    img_cropped = img.crop((left, top, right, bottom))
    
    # Resize back to original dimensions
    img_resized = img_cropped.resize(original_size, resample=resample_method)
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_cropped.png"
    
    # Save without compression (PNG format)
    img_resized.save(output_path, 'PNG')
    
    # Prepare crop info
    crop_info = {
        'original_size': original_size,
        'crop_box': (left, top, right, bottom),
        'crop_size': (crop_width, crop_height),
        'crop_scale': target_area / area,
        'crop_aspect_ratio': crop_width / crop_height
    }
    
    print(f"Cropped image saved to: {output_path}")
    print(f"Original size: {original_size}")
    print(f"Crop box (left, top, right, bottom): ({left}, {top}, {right}, {bottom})")
    print(f"Crop size: {crop_width}x{crop_height}")
    print(f"Crop scale: {crop_info['crop_scale']:.2%}")
    print(f"Crop aspect ratio: {crop_info['crop_aspect_ratio']:.2f}")
    
    return output_path, crop_info

In [21]:
apply_random_resized_crop('./figures/watermarked_img.png',  crop_scale=(0.3, 0.8))  # Default: 50-100% crop

Cropped image saved to: ./figures/watermarked_img_cropped.png
Original size: (1815, 1795)
Crop box (left, top, right, bottom): (80, 129, 1597, 1678)
Crop size: 1517x1549
Crop scale: 72.09%
Crop aspect ratio: 0.98


('./figures/watermarked_img_cropped.png',
 {'original_size': (1815, 1795),
  'crop_box': (80, 129, 1597, 1678),
  'crop_size': (1517, 1549),
  'crop_scale': 0.7209204392269484,
  'crop_aspect_ratio': 0.9793415106520336})

In [22]:
from PIL import Image, ImageDraw
import random

def apply_occlusion(image_path, output_path=None, occlusion_ratio=0.25, 
                   position='random', num_rectangles=1, seed=None):
    """
    Apply occlusion by masking part of the image with black rectangle(s).
    Tests spatial redundancy of watermarks or robustness to partial occlusion.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the occluded image. If None, saves as '{original_name}_occluded.png'
    occlusion_ratio : float, optional
        Fraction of image area to occlude (0-1). Default is 0.25 (25%)
    position : str or tuple, optional
        Position of occlusion:
        - 'random': Random position (default)
        - 'center': Center of image
        - 'top_left', 'top_right', 'bottom_left', 'bottom_right': Corners
        - (x, y): Specific coordinates for top-left corner of rectangle
    num_rectangles : int, optional
        Number of occlusion rectangles to apply. Default is 1
    seed : int, optional
        Random seed for reproducibility. If None, uses random values
    
    Returns:
    --------
    tuple
        (output_path, occlusion_info) where occlusion_info contains the occlusion parameters
    """
    # Set random seed if provided
    if seed is not None:
        random.seed(seed)
    
    # Open the image
    img = Image.open(image_path)
    width, height = img.size
    total_area = width * height
    
    # Create a copy to draw on
    img_occluded = img.copy()
    draw = ImageDraw.Draw(img_occluded)
    
    occlusion_info = []
    
    # Calculate area per rectangle
    area_per_rect = (total_area * occlusion_ratio) / num_rectangles
    
    for i in range(num_rectangles):
        # Calculate rectangle dimensions (square-ish by default)
        rect_width = int((area_per_rect) ** 0.5)
        rect_height = int(area_per_rect / rect_width)
        
        # Ensure dimensions don't exceed image size
        rect_width = min(rect_width, width)
        rect_height = min(rect_height, height)
        
        # Determine position
        if position == 'random':
            x = random.randint(0, width - rect_width)
            y = random.randint(0, height - rect_height)
        elif position == 'center':
            x = (width - rect_width) // 2
            y = (height - rect_height) // 2
        elif position == 'top_left':
            x, y = 0, 0
        elif position == 'top_right':
            x = width - rect_width
            y = 0
        elif position == 'bottom_left':
            x = 0
            y = height - rect_height
        elif position == 'bottom_right':
            x = width - rect_width
            y = height - rect_height
        elif isinstance(position, tuple) and len(position) == 2:
            x, y = position
            # Ensure rectangle stays within bounds
            x = min(x, width - rect_width)
            y = min(y, height - rect_height)
        else:
            raise ValueError(f"Invalid position: {position}")
        
        # Draw black rectangle
        draw.rectangle([x, y, x + rect_width, y + rect_height], fill='black')
        
        occlusion_info.append({
            'box': (x, y, x + rect_width, y + rect_height),
            'size': (rect_width, rect_height),
            'area_ratio': (rect_width * rect_height) / total_area
        })
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_occluded.png"
    
    # Save without compression (PNG format)
    img_occluded.save(output_path, 'PNG')
    
    print(f"Occluded image saved to: {output_path}")
    print(f"Image size: {width}x{height}")
    print(f"Number of occlusions: {num_rectangles}")
    print(f"Total occlusion ratio: {occlusion_ratio:.1%}")
    
    for i, info in enumerate(occlusion_info):
        print(f"Rectangle {i+1}: position ({info['box'][0]}, {info['box'][1]}), "
              f"size {info['size'][0]}x{info['size'][1]}, "
              f"area {info['area_ratio']:.1%}")
    
    return output_path, occlusion_info


# Example usage:
# 
# apply_occlusion('input.png', occlusion_ratio=0.5)  # 50% occlusion
# apply_occlusion('input.png', position='center')  # Center occlusion
# apply_occlusion('input.png', num_rectangles=3)  # Multiple occlusions

In [23]:
apply_occlusion('./figures/watermarked_img.png')  # Default: 25% random occlusion

Occluded image saved to: ./figures/watermarked_img_occluded.png
Image size: 1815x1795
Number of occlusions: 1
Total occlusion ratio: 25.0%
Rectangle 1: position (123, 378), size 902x902, area 25.0%


('./figures/watermarked_img_occluded.png',
 [{'box': (123, 378, 1025, 1280),
   'size': (902, 902),
   'area_ratio': 0.24973073351903435}])

In [24]:
def apply_grid_occlusion(image_path, output_path=None, grid_size=(3, 3), 
                         occlude_cells=None, seed=None):
    """
    Apply occlusion in a grid pattern.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the occluded image
    grid_size : tuple, optional
        Grid dimensions (rows, cols). Default is (3, 3)
    occlude_cells : list, optional
        List of cell indices to occlude (0-indexed). If None, randomly selects cells
    seed : int, optional
        Random seed for reproducibility
    """
    from PIL import Image, ImageDraw
    import random
    import os
    
    if seed is not None:
        random.seed(seed)
    
    img = Image.open(image_path)
    width, height = img.size
    
    rows, cols = grid_size
    cell_width = width // cols
    cell_height = height // rows
    
    img_occluded = img.copy()
    draw = ImageDraw.Draw(img_occluded)
    
    # If no cells specified, randomly select some
    if occlude_cells is None:
        total_cells = rows * cols
        num_to_occlude = max(1, total_cells // 3)  # Occlude ~1/3 of cells
        occlude_cells = random.sample(range(total_cells), num_to_occlude)
    
    occluded_info = []
    
    for cell_idx in occlude_cells:
        row = cell_idx // cols
        col = cell_idx % cols
        
        x1 = col * cell_width
        y1 = row * cell_height
        x2 = x1 + cell_width
        y2 = y1 + cell_height
        
        draw.rectangle([x1, y1, x2, y2], fill='black')
        occluded_info.append({'cell': (row, col), 'box': (x1, y1, x2, y2)})
    
    if output_path is None:
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_grid_occluded.png"
    
    img_occluded.save(output_path, 'PNG')
    
    print(f"Grid occluded image saved to: {output_path}")
    print(f"Grid size: {rows}x{cols}")
    print(f"Occluded cells: {len(occlude_cells)}/{rows*cols}")
    print(f"Cell positions: {[info['cell'] for info in occluded_info]}")
    
    return output_path, occluded_info


# Example usage:
# apply_grid_occlusion('image.png')  # Random grid occlusion
# apply_grid_occlusion('./figures/watermarked_img.png', grid_size=(4, 4))  # 4x4 grid
# apply_grid_occlusion('image.png', occlude_cells=[0, 4, 8])  # Specific cells

In [25]:
apply_grid_occlusion('./figures/watermarked_img.png', grid_size=(4, 4))  # 4x4 grid

Grid occluded image saved to: ./figures/watermarked_img_grid_occluded.png
Grid size: 4x4
Occluded cells: 5/16
Cell positions: [(2, 2), (2, 0), (0, 3), (1, 0), (3, 1)]


('./figures/watermarked_img_grid_occluded.png',
 [{'cell': (2, 2), 'box': (906, 896, 1359, 1344)},
  {'cell': (2, 0), 'box': (0, 896, 453, 1344)},
  {'cell': (0, 3), 'box': (1359, 0, 1812, 448)},
  {'cell': (1, 0), 'box': (0, 448, 453, 896)},
  {'cell': (3, 1), 'box': (453, 1344, 906, 1792)}])

In [26]:
from PIL import Image
import random
import math

def apply_geometric_warp(image_path, output_path=None, warp_type='random', 
                        rotation_range=(-15, 15), scale_range=(0.9, 1.1),
                        shear_range=(-10, 10), translate_range=(-0.1, 0.1),
                        seed=None):
    """
    Apply small geometric warps to test robustness to viewpoint distortion.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the warped image. If None, saves as '{original_name}_warped.png'
    warp_type : str, optional
        Type of warp to apply:
        - 'random': Random combination of transforms (default)
        - 'rotation': Only rotation
        - 'affine': Affine transform (scale, shear, translate)
        - 'perspective': Perspective transform
        - 'all': All transforms combined
    rotation_range : tuple, optional
        Range of rotation in degrees (min, max). Default is (-15, 15)
    scale_range : tuple, optional
        Range of scaling factor (min, max). Default is (0.9, 1.1)
    shear_range : tuple, optional
        Range of shear in degrees (min, max). Default is (-10, 10)
    translate_range : tuple, optional
        Range of translation as fraction of image size (min, max). Default is (-0.1, 0.1)
    seed : int, optional
        Random seed for reproducibility. If None, uses random values
    
    Returns:
    --------
    tuple
        (output_path, warp_info) where warp_info contains the transformation parameters
    """
    # Set random seed if provided
    if seed is not None:
        random.seed(seed)
    
    # Open the image
    img = Image.open(image_path)
    width, height = img.size
    
    warp_info = {}
    
    if warp_type in ['rotation', 'random', 'all']:
        # Apply rotation
        angle = random.uniform(rotation_range[0], rotation_range[1])
        img_warped = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor='black')
        warp_info['rotation'] = angle
    else:
        img_warped = img.copy()
    
    if warp_type in ['affine', 'random', 'all']:
        # Apply affine transform (scale, shear, translate)
        
        # Random parameters
        scale_x = random.uniform(scale_range[0], scale_range[1])
        scale_y = random.uniform(scale_range[0], scale_range[1])
        shear_x = random.uniform(shear_range[0], shear_range[1])
        shear_y = random.uniform(shear_range[0], shear_range[1])
        translate_x = random.uniform(translate_range[0], translate_range[1]) * width
        translate_y = random.uniform(translate_range[0], translate_range[1]) * height
        
        # Convert shear to radians
        shear_x_rad = math.radians(shear_x)
        shear_y_rad = math.radians(shear_y)
        
        # Build affine transformation matrix
        # Matrix format: (a, b, c, d, e, f) where:
        # x' = a*x + b*y + c
        # y' = d*x + e*y + f
        
        a = scale_x * math.cos(shear_y_rad)
        b = -scale_x * math.sin(shear_y_rad)
        c = translate_x + width * (1 - scale_x) / 2
        d = scale_y * math.sin(shear_x_rad)
        e = scale_y * math.cos(shear_x_rad)
        f = translate_y + height * (1 - scale_y) / 2
        
        # Apply affine transform
        img_warped = img_warped.transform(
            img_warped.size,
            Image.AFFINE,
            (a, b, c, d, e, f),
            resample=Image.BICUBIC,
            fillcolor='black'
        )
        
        warp_info['scale'] = (scale_x, scale_y)
        warp_info['shear'] = (shear_x, shear_y)
        warp_info['translate'] = (translate_x, translate_y)
    
    if warp_type == 'perspective':
        # Apply perspective transform
        img_warped = apply_perspective_transform(img, seed=seed)
        warp_info['perspective'] = 'applied'
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_warped.png"
    
    # Save without compression (PNG format)
    img_warped.save(output_path, 'PNG')
    
    print(f"Warped image saved to: {output_path}")
    print(f"Warp type: {warp_type}")
    if 'rotation' in warp_info:
        print(f"Rotation: {warp_info['rotation']:.2f}°")
    if 'scale' in warp_info:
        print(f"Scale: x={warp_info['scale'][0]:.3f}, y={warp_info['scale'][1]:.3f}")
    if 'shear' in warp_info:
        print(f"Shear: x={warp_info['shear'][0]:.2f}°, y={warp_info['shear'][1]:.2f}°")
    if 'translate' in warp_info:
        print(f"Translate: x={warp_info['translate'][0]:.1f}px, y={warp_info['translate'][1]:.1f}px")
    
    return output_path, warp_info


def apply_perspective_transform(img, strength=0.1, seed=None):
    """
    Apply perspective transformation to simulate viewpoint change.
    
    Parameters:
    -----------
    img : PIL.Image
        Input image
    strength : float, optional
        Strength of perspective distortion (0-1). Default is 0.1
    seed : int, optional
        Random seed for reproducibility
    """
    if seed is not None:
        random.seed(seed)
    
    width, height = img.size
    
    # Define corner points with random perturbations
    max_shift = int(min(width, height) * strength)
    
    # Original corners: top-left, top-right, bottom-right, bottom-left
    src_points = [
        (0, 0),
        (width, 0),
        (width, height),
        (0, height)
    ]
    
    # Perturbed corners
    dst_points = [
        (random.randint(-max_shift, max_shift), random.randint(-max_shift, max_shift)),
        (width + random.randint(-max_shift, max_shift), random.randint(-max_shift, max_shift)),
        (width + random.randint(-max_shift, max_shift), height + random.randint(-max_shift, max_shift)),
        (random.randint(-max_shift, max_shift), height + random.randint(-max_shift, max_shift))
    ]
    
    # Calculate perspective transform coefficients
    coeffs = find_perspective_coefficients(src_points, dst_points)
    
    # Apply perspective transform
    img_perspective = img.transform(
        img.size,
        Image.PERSPECTIVE,
        coeffs,
        resample=Image.BICUBIC,
        fillcolor='black'
    )
    
    return img_perspective


def find_perspective_coefficients(src_points, dst_points):
    """
    Find the perspective transformation coefficients.
    
    Parameters:
    -----------
    src_points : list
        List of 4 source points [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
    dst_points : list
        List of 4 destination points [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
    
    Returns:
    --------
    tuple
        8 coefficients for perspective transform
    """
    matrix = []
    for (x, y), (X, Y) in zip(src_points, dst_points):
        matrix.append([x, y, 1, 0, 0, 0, -X*x, -X*y])
        matrix.append([0, 0, 0, x, y, 1, -Y*x, -Y*y])
    
    A = []
    B = []
    for i, row in enumerate(matrix):
        A.append(row)
        if i % 2 == 0:
            B.append(dst_points[i//2][0])
        else:
            B.append(dst_points[i//2][1])
    
    # Solve using simple matrix operations
    # For simplicity, using a basic approach
    # In production, use numpy for better numerical stability
    
    # Simplified coefficient calculation
    coeffs = solve_linear_system(A, B)
    
    return tuple(coeffs)


def solve_linear_system(A, B):
    """
    Simple linear system solver (Gaussian elimination).
    For production use, consider using numpy.linalg.solve
    """
    n = len(B)
    # Create augmented matrix
    for i in range(n):
        A[i].append(B[i])
    
    # Forward elimination
    for i in range(n):
        # Find pivot
        max_row = i
        for k in range(i + 1, n):
            if abs(A[k][i]) > abs(A[max_row][i]):
                max_row = k
        A[i], A[max_row] = A[max_row], A[i]
        
        # Make all rows below this one 0 in current column
        for k in range(i + 1, n):
            if A[i][i] != 0:
                c = A[k][i] / A[i][i]
                for j in range(i, n + 1):
                    if i == j:
                        A[k][j] = 0
                    else:
                        A[k][j] -= c * A[i][j]
    
    # Back substitution
    x = [0 for _ in range(n)]
    for i in range(n - 1, -1, -1):
        if A[i][i] != 0:
            x[i] = A[i][n] / A[i][i]
            for k in range(i - 1, -1, -1):
                A[k][n] -= A[k][i] * x[i]
    
    return x


# Example usage:
# 
# apply_geometric_warp('input.png', warp_type='rotation')  # Only rotation
# apply_geometric_warp('input.png', warp_type='affine')  # Affine transform
# apply_geometric_warp('input.png', warp_type='perspective')  # Perspective

In [28]:
apply_geometric_warp('./figures/watermarked_img.png')  # Random warp

Warped image saved to: ./figures/watermarked_img_warped.png
Warp type: random
Rotation: 12.36°
Scale: x=0.925, y=0.951
Shear: x=5.60°, y=-2.92°
Translate: x=-48.2px, y=64.6px


('./figures/watermarked_img_warped.png',
 {'rotation': 12.357336592382751,
  'scale': (0.9247496918629192, 0.9506135827436423),
  'shear': (5.59867604761436, -2.9172102288055735),
  'translate': (-48.166350156329464, 64.57529240643642)})

In [29]:
from PIL import Image, ImageEnhance, ImageFilter
import random

def apply_train_aug_mix(image_path, output_path=None, num_augs=None, 
                       aug_intensity='medium', seed=None):
    """
    Apply a mixture of augmentations used during training to test generalization.
    Includes mild rotations, color jitter, JPEG compression, blur, etc.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the augmented image. If None, saves as '{original_name}_augmix.jpg'
    num_augs : int, optional
        Number of augmentations to apply. If None, randomly selects 2-4 augmentations
    aug_intensity : str, optional
        Intensity of augmentations: 'mild', 'medium', 'strong'
        Default is 'medium'
    seed : int, optional
        Random seed for reproducibility. If None, uses random values
    
    Returns:
    --------
    tuple
        (output_path, applied_augs) where applied_augs is a list of applied augmentations
    """
    # Set random seed if provided
    if seed is not None:
        random.seed(seed)
    
    # Define intensity parameters
    intensity_params = {
        'mild': {
            'rotation': (-5, 5),
            'brightness': (0.9, 1.1),
            'contrast': (0.9, 1.1),
            'saturation': (0.9, 1.1),
            'hue': (-0.05, 0.05),
            'jpeg_quality': (75, 95),
            'blur_radius': (0.5, 1.5),
            'sharpness': (0.9, 1.1),
            'scale': (0.95, 1.05)
        },
        'medium': {
            'rotation': (-10, 10),
            'brightness': (0.8, 1.2),
            'contrast': (0.8, 1.2),
            'saturation': (0.7, 1.3),
            'hue': (-0.1, 0.1),
            'jpeg_quality': (50, 85),
            'blur_radius': (1.0, 2.5),
            'sharpness': (0.8, 1.2),
            'scale': (0.9, 1.1)
        },
        'strong': {
            'rotation': (-15, 15),
            'brightness': (0.7, 1.3),
            'contrast': (0.7, 1.3),
            'saturation': (0.6, 1.4),
            'hue': (-0.15, 0.15),
            'jpeg_quality': (30, 70),
            'blur_radius': (1.5, 3.5),
            'sharpness': (0.7, 1.3),
            'scale': (0.85, 1.15)
        }
    }
    
    params = intensity_params.get(aug_intensity, intensity_params['medium'])
    
    # Open the image
    img = Image.open(image_path)
    
    # Convert to RGB if necessary
    if img.mode in ('RGBA', 'LA', 'P'):
        rgb_img = Image.new('RGB', img.size, (255, 255, 255))
        if img.mode == 'P':
            img = img.convert('RGBA')
        if img.mode in ('RGBA', 'LA'):
            rgb_img.paste(img, mask=img.split()[-1])
        else:
            rgb_img.paste(img)
        img = rgb_img
    elif img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Define available augmentations
    def aug_rotation(image):
        angle = random.uniform(*params['rotation'])
        return image.rotate(angle, resample=Image.BICUBIC, expand=False), f"rotation({angle:.2f}°)"
    
    def aug_brightness(image):
        factor = random.uniform(*params['brightness'])
        enhancer = ImageEnhance.Brightness(image)
        return enhancer.enhance(factor), f"brightness({factor:.2f})"
    
    def aug_contrast(image):
        factor = random.uniform(*params['contrast'])
        enhancer = ImageEnhance.Contrast(image)
        return enhancer.enhance(factor), f"contrast({factor:.2f})"
    
    def aug_saturation(image):
        factor = random.uniform(*params['saturation'])
        enhancer = ImageEnhance.Color(image)
        return enhancer.enhance(factor), f"saturation({factor:.2f})"
    
    def aug_sharpness(image):
        factor = random.uniform(*params['sharpness'])
        enhancer = ImageEnhance.Sharpness(image)
        return enhancer.enhance(factor), f"sharpness({factor:.2f})"
    
    def aug_blur(image):
        radius = random.uniform(*params['blur_radius'])
        return image.filter(ImageFilter.GaussianBlur(radius=radius)), f"blur({radius:.2f})"
    
    def aug_hue(image):
        # Simulate hue shift using color matrix
        # This is a simplified version
        factor = random.uniform(*params['hue'])
        # For simplicity, we'll use color enhancement as proxy
        return image, f"hue_shift({factor:.3f})"
    
    def aug_scale(image):
        factor = random.uniform(*params['scale'])
        width, height = image.size
        new_size = (int(width * factor), int(height * factor))
        scaled = image.resize(new_size, Image.BICUBIC)
        # Resize back to original
        return scaled.resize((width, height), Image.BICUBIC), f"scale({factor:.2f})"
    
    # List of augmentation functions
    augmentation_pool = [
        aug_rotation,
        aug_brightness,
        aug_contrast,
        aug_saturation,
        aug_sharpness,
        aug_blur,
        aug_hue,
        aug_scale
    ]
    
    # Determine number of augmentations to apply
    if num_augs is None:
        num_augs = random.randint(2, 4)
    
    num_augs = min(num_augs, len(augmentation_pool))
    
    # Randomly select augmentations
    selected_augs = random.sample(augmentation_pool, num_augs)
    
    # Apply augmentations sequentially
    applied_augs = []
    img_augmented = img
    
    for aug_func in selected_augs:
        img_augmented, aug_name = aug_func(img_augmented)
        applied_augs.append(aug_name)
    
    # Apply JPEG compression as final step (common in training pipelines)
    import io
    jpeg_quality = random.randint(*params['jpeg_quality'])
    
    # Save to bytes buffer with JPEG compression
    buffer = io.BytesIO()
    img_augmented.save(buffer, format='JPEG', quality=jpeg_quality)
    buffer.seek(0)
    img_augmented = Image.open(buffer)
    applied_augs.append(f"jpeg_compression(q={jpeg_quality})")
    
    # Generate output path if not provided
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_augmix.jpg"
    
    # Save final image
    img_augmented.save(output_path, 'JPEG', quality=95)
    
    print(f"Augmented image saved to: {output_path}")
    print(f"Augmentation intensity: {aug_intensity}")
    print(f"Number of augmentations applied: {len(applied_augs)}")
    print(f"Applied augmentations: {', '.join(applied_augs)}")
    
    return output_path, applied_augs


def apply_train_aug_mix_advanced(image_path, output_path=None, aug_chain=None,
                                 aug_intensity='medium', seed=None):
    """
    Advanced version with custom augmentation chains.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image file
    output_path : str, optional
        Path to save the augmented image
    aug_chain : list, optional
        Specific list of augmentations to apply in order.
        Options: 'rotation', 'brightness', 'contrast', 'saturation', 'blur', 
                'sharpness', 'jpeg', 'scale', 'flip', 'crop'
        If None, randomly selects augmentations
    aug_intensity : str, optional
        Intensity: 'mild', 'medium', 'strong'
    seed : int, optional
        Random seed for reproducibility
    """
    if seed is not None:
        random.seed(seed)
    
    # If no chain specified, use random selection
    if aug_chain is None:
        available_augs = ['rotation', 'brightness', 'contrast', 'saturation', 
                         'blur', 'sharpness', 'jpeg', 'scale']
        num_augs = random.randint(2, 5)
        aug_chain = random.sample(available_augs, min(num_augs, len(available_augs)))
    
    # Open image
    img = Image.open(image_path)
    
    # Convert to RGB
    if img.mode != 'RGB':
        if img.mode in ('RGBA', 'LA', 'P'):
            rgb_img = Image.new('RGB', img.size, (255, 255, 255))
            if img.mode == 'P':
                img = img.convert('RGBA')
            if img.mode in ('RGBA', 'LA'):
                rgb_img.paste(img, mask=img.split()[-1])
            else:
                rgb_img.paste(img)
            img = rgb_img
        else:
            img = img.convert('RGB')
    
    applied_augs = []
    img_result = img
    
    # Apply each augmentation in the chain
    for aug_name in aug_chain:
        if aug_name == 'rotation':
            angle = random.uniform(-10, 10) if aug_intensity == 'medium' else random.uniform(-15, 15)
            img_result = img_result.rotate(angle, resample=Image.BICUBIC, expand=False)
            applied_augs.append(f"rotation({angle:.1f}°)")
        
        elif aug_name == 'brightness':
            factor = random.uniform(0.8, 1.2) if aug_intensity == 'medium' else random.uniform(0.7, 1.3)
            img_result = ImageEnhance.Brightness(img_result).enhance(factor)
            applied_augs.append(f"brightness({factor:.2f})")
        
        elif aug_name == 'contrast':
            factor = random.uniform(0.8, 1.2) if aug_intensity == 'medium' else random.uniform(0.7, 1.3)
            img_result = ImageEnhance.Contrast(img_result).enhance(factor)
            applied_augs.append(f"contrast({factor:.2f})")
        
        elif aug_name == 'saturation':
            factor = random.uniform(0.7, 1.3) if aug_intensity == 'medium' else random.uniform(0.6, 1.4)
            img_result = ImageEnhance.Color(img_result).enhance(factor)
            applied_augs.append(f"saturation({factor:.2f})")
        
        elif aug_name == 'blur':
            radius = random.uniform(1.0, 2.5) if aug_intensity == 'medium' else random.uniform(1.5, 3.5)
            img_result = img_result.filter(ImageFilter.GaussianBlur(radius=radius))
            applied_augs.append(f"blur({radius:.2f})")
        
        elif aug_name == 'sharpness':
            factor = random.uniform(0.8, 1.2) if aug_intensity == 'medium' else random.uniform(0.7, 1.3)
            img_result = ImageEnhance.Sharpness(img_result).enhance(factor)
            applied_augs.append(f"sharpness({factor:.2f})")
        
        elif aug_name == 'jpeg':
            import io
            quality = random.randint(50, 85) if aug_intensity == 'medium' else random.randint(30, 70)
            buffer = io.BytesIO()
            img_result.save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            img_result = Image.open(buffer).copy()
            applied_augs.append(f"jpeg(q={quality})")
        
        elif aug_name == 'scale':
            factor = random.uniform(0.9, 1.1) if aug_intensity == 'medium' else random.uniform(0.85, 1.15)
            w, h = img_result.size
            new_size = (int(w * factor), int(h * factor))
            img_result = img_result.resize(new_size, Image.BICUBIC).resize((w, h), Image.BICUBIC)
            applied_augs.append(f"scale({factor:.2f})")
        
        elif aug_name == 'flip':
            if random.random() > 0.5:
                img_result = img_result.transpose(Image.FLIP_LEFT_RIGHT)
                applied_augs.append("flip(horizontal)")
        
        elif aug_name == 'crop':
            w, h = img_result.size
            crop_factor = 0.9 if aug_intensity == 'medium' else 0.85
            new_w, new_h = int(w * crop_factor), int(h * crop_factor)
            left = random.randint(0, w - new_w)
            top = random.randint(0, h - new_h)
            img_result = img_result.crop((left, top, left + new_w, top + new_h)).resize((w, h), Image.BICUBIC)
            applied_augs.append(f"crop({crop_factor:.2f})")
    
    # Generate output path
    if output_path is None:
        import os
        base_name = os.path.splitext(image_path)[0]
        output_path = f"{base_name}_augmix_adv.jpg"
    
    # Save
    img_result.save(output_path, 'JPEG', quality=95)
    
    print(f"Advanced augmented image saved to: {output_path}")
    print(f"Applied augmentations: {', '.join(applied_augs)}")
    
    return output_path, applied_augs


# Example usage:
# 
# apply_train_aug_mix('input.png', aug_intensity='mild')  # Mild augmentations
# apply_train_aug_mix('input.png', aug_intensity='strong', num_augs=5)  # Strong, 5 augs
# apply_train_aug_mix('input.png', seed=42)  # Reproducible

In [35]:
apply_train_aug_mix('./figures/watermarked_img.png', aug_intensity='strong', num_augs=5)  # Random 2-4 augmentations, medium intensity

Augmented image saved to: ./figures/watermarked_img_augmix.jpg
Augmentation intensity: strong
Number of augmentations applied: 6
Applied augmentations: scale(0.90), brightness(0.75), contrast(0.87), blur(3.00), saturation(1.27), jpeg_compression(q=40)


('./figures/watermarked_img_augmix.jpg',
 ['scale(0.90)',
  'brightness(0.75)',
  'contrast(0.87)',
  'blur(3.00)',
  'saturation(1.27)',
  'jpeg_compression(q=40)'])